In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

from peft import PeftModel

## 加载基础模型

In [2]:
model = AutoModelForCausalLM.from_pretrained("../../model/langboat/bloom-1b4-zh")
tokenizer = AutoTokenizer.from_pretrained("../../model/langboat/bloom-1b4-zh")

## 加载Lora模型

In [3]:
p_model = PeftModel.from_pretrained(model, model_id="./chatbot/checkpoint-3357/")
p_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): BloomForCausalLM(
      (transformer): BloomModel(
        (word_embeddings): Embedding(46145, 2048)
        (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (h): ModuleList(
          (0): BloomBlock(
            (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (self_attention): BloomAttention(
              (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
              (dense): Linear(in_features=2048, out_features=2048, bias=True)
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
            (mlp): BloomMLP(
              (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
              (gelu_impl): BloomGelu()
              (dense_4h_to_h): Linear(in_features=8192, out_feature

In [5]:
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt")
tokenizer.decode(p_model.generate(**ipt, do_sample=False, max_length=100)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧有很多，比如：\n1. 提前做好准备：提前做好准备，可以有效提高考试的效率。例如，提前阅读试卷，了解试题的难易程度，熟悉答题流程，提前做好答题的准备。\n2. 合理分配时间：合理分配时间，可以有效提高考试的效率。例如，合理安排答题时间，避免长时间思考，提高答题速度。\n'

## 模型合并

In [6]:
merge_model = p_model.merge_and_unload()
merge_model

BloomForCausalLM(
  (transformer): BloomModel(
    (word_embeddings): Embedding(46145, 2048)
    (word_embeddings_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
    (h): ModuleList(
      (0-23): 24 x BloomBlock(
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (self_attention): BloomAttention(
          (query_key_value): Linear(in_features=2048, out_features=6144, bias=True)
          (dense): Linear(in_features=2048, out_features=2048, bias=True)
          (attention_dropout): Dropout(p=0.0, inplace=False)
        )
        (post_attention_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (mlp): BloomMLP(
          (dense_h_to_4h): Linear(in_features=2048, out_features=8192, bias=True)
          (gelu_impl): BloomGelu()
          (dense_4h_to_h): Linear(in_features=8192, out_features=2048, bias=True)
        )
      )
    )
    (ln_f): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
  )
  (l

In [8]:
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt")
tokenizer.decode(merge_model.generate(**ipt, do_sample=False, max_length=100)[0], skip_special_tokens=True)

'Human: 考试有哪些技巧？\n\nAssistant: 考试技巧有很多，比如：\n1. 提前做好准备：提前做好准备，可以有效提高考试的效率。例如，提前阅读试卷，了解试题的难易程度，熟悉答题流程，提前做好答题的准备。\n2. 合理分配时间：合理分配时间，可以有效提高考试的效率。例如，合理安排答题时间，避免长时间思考，提高答题速度。\n'

## 完整模型保存

In [ ]:
merge_model.save_pretrained("./chatbot/merge_model")